In [1]:
# asyncpg: 비동기, httpx: 외부 API(http)비동기 호출
%pip install asyncpg httpx python-dotenv
%pip install requests


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# 테스트용 초기 state 만들기
from core.state import make_initial_state
from core.mocks import mock_user_input

initial_state = make_initial_state(mock_user_input)

In [4]:
# [노드] validate_input
# 사용자 입력 -> 좌표 변환, 식사 시간 계산 (전처리)
from nodes.validate_input import validate_input

validate_result = validate_input(initial_state)
# print("validate_input 결과:", validate_result)


In [5]:
# [노드] collect_candidate_pool
# kakao Local API로 raw 후보 풀 수집 + PostgreSQL
from nodes.collect_candidate_pool import collect_candidate_pool

candidates_result = await collect_candidate_pool(validate_result)
#print("candidates_result 결과:", candidates_result)


In [6]:
# [노드] first_filter_candidates
# 첫번쨰 필터 -> 50개 축소
from nodes.first_filter_candidates import first_filter_candidates

filter_result = first_filter_candidates(candidates_result, debug=True)

print(f"\n📌 warnings:")
for w in filter_result["warnings"]:
    print(f"   - {w}")


📦 시작
   ──────────────────────────────────────────────────
   ✂️  제거: 0개  |  남은: 275개
   📊 카테고리: :110  FD6:68  CE7:59  AD5:12  AT4:9  PK6:7
🔍 전체 장소 목록:
01. [CE7] 오프온 | 음식점 > 카페
02. [CE7] 카페해운대1994 | 음식점 > 카페
03. [CE7] 부산바다샌드 해운대본점 | 음식점 > 카페 > 테마카페 > 디저트카페
04. [CE7] 카페히토 | 음식점 > 카페
05. [CE7] 블랙업커피 해운대점 | 음식점 > 카페 > 커피전문점
06. [CE7] 듀플릿 | 음식점 > 카페 > 테마카페 > 디저트카페
07. [CE7] 금송덕미 해리단길점 | 음식점 > 카페 > 테마카페 > 디저트카페
08. [CE7] 브레이크아웃이스케이프 해운대본점 | 음식점 > 카페 > 테마카페
09. [CE7] 투썸플레이스 해운대센트럴푸르지오점 | 음식점 > 카페 > 커피전문점 > 투썸플레이스
10. [CE7] 코오리마찌 | 음식점 > 카페 > 테마카페 > 디저트카페
11. [CE7] 모루씨 해리단길점 | 음식점 > 카페 > 커피전문점
12. [CE7] 로우앤스윗 로스터리 | 음식점 > 카페
13. [CE7] 제스터 | 음식점 > 카페
14. [CE7] 하브커피 | 음식점 > 카페 > 커피전문점
15. [CE7] 워킹홀리데이 해운대 | 음식점 > 카페
16. [CE7] 홈즈앤루팡 해운대점 | 가정,생활 > 여가시설 > 보드카페 > 홈즈앤루팡
17. [CE7] 로우앤스윗 해운대카페 | 음식점 > 카페 > 커피전문점
18. [CE7] 해운대옛날팥빙수단팥죽 | 음식점 > 카페 > 테마카페 > 디저트카페
19. [CE7] 프라한 해운대점 | 음식점 > 카페 > 테마카페
20. [CE7] 스타벅스 하버타운점 | 음식점 > 카페 > 커피전문점 > 스타벅스
21. [CE7] 엡섬 | 음식점 > 카페
22. [CE7] 카페플럼피 | 음식점 > 카페 > 커피전문점

In [ ]:
# [노드] second_filter_candidates
# 두번쨰 필터 -> 30개 축소
from nodes.second_filter_candidates import second_filter_candidates

test_state = {
    "filtered_candidates": filter_result["filtered_candidates"],
    "user_input": mock_user_input,
    "warnings": [],
}

second_filter_result = await second_filter_candidates(test_state)

print(f"⚠️ warnings: {second_filter_result['warnings']}")
print(f"✅ step: {second_filter_result['step']}")
print(f"📦 보강된 장소 수: {len(second_filter_result['filtered_candidates'])}")
print(f"📊 scored_candidates 수: {len(second_filter_result['scored_candidates'])}")
print(f"🏆 shortlist 수: {len(second_filter_result['shortlist'])}")

print("\n=== 전체 scored_candidates (50개) ===")

for i, s in enumerate(second_filter_result["scored_candidates"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  분위기: {p.get('atmosphere')}
  추천대상: {p.get('best_for')}
  활동: {p.get('place_tags')}
  재방문의사: {p.get('revisit_intent')}
  한줄요약: {p.get('summary')}
  mood_score: {s['mood_score']}
  activity_score: {s['activity_score']}
  party_fit_score: {s['party_fit_score']}
  total_score: {s['total_score']}
""")


print("\n=== shortlist (최종 30개) ===")

for i, s in enumerate(second_filter_result["shortlist"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  total_score: {s['total_score']}
  bucket: {p.get('bucket')}
""")

In [ ]:
# [노드] travel_matrix
# shortlist 30개 장소 간 이동시간 행렬 계산
from nodes.travel_matrix import travel_matrix

matrix_result = travel_matrix(second_filter_result)

# place_id → name 매핑
id_to_name = {item["place"]["id"]: item["place"]["name"] for item in second_filter_result["shortlist"]}

print(f"✅ place_index: {len(matrix_result['place_index'])}개")
print(f"✅ distance_matrix: {len(matrix_result['distance_matrix'])}x{len(matrix_result['distance_matrix'][0])}")
print(f"✅ time_matrix: {len(matrix_result['time_matrix'])}x{len(matrix_result['time_matrix'][0])}")

for i, from_id in enumerate(matrix_result['place_index']):
    print(f"\n📍 {id_to_name[from_id]} 에서:")
    for to_id, mins in zip(matrix_result['place_index'], matrix_result['time_matrix'][i]):
        if from_id == to_id:
            continue
        print(f"  → {id_to_name[to_id]}: {mins}분")

In [ ]:
# [노드] plan_itinerary
from nodes.plan_itinerary import plan_itinerary

# travel_matrix 결과 + user_input 합치기
plan_state = {**matrix_result, "shortlist": second_filter_result["shortlist"], "user_input": second_filter_result["user_input"]}
itinerary_result = plan_itinerary(plan_state)

print(f"✅ 총 {len(itinerary_result['itinerary'])}개 장소")
print()
for item in itinerary_result["itinerary"]:
    p = item["place"]
    print(f"{item['order']}. [{p.get('bucket','?'):8}] {p['name']}")
    print(f"   도착: {item['arrive_at']}  출발: {item['leave_at']}  다음까지: {item['travel_to_next_minutes']}분")
    print()

In [ ]:
# LangGraph 그래프 빌드
from langgraph.graph import StateGraph, START, END
from core.state import TravelState

graph_builder = StateGraph(TravelState)

# 노드 등록
graph_builder.add_node("validate_input", validate_input)
graph_builder.add_node("collect_candidate_pool", collect_candidate_pool)
graph_builder.add_node("first_filter_candidates", first_filter_candidates)
graph_builder.add_node("second_filter_candidates", second_filter_candidates)
graph_builder.add_node("travel_matrix", travel_matrix)
graph_builder.add_node("plan_itinerary", plan_itinerary)

# 엣지 (직선 연결)
graph_builder.add_edge(START, "validate_input")
graph_builder.add_edge("validate_input", "collect_candidate_pool")
graph_builder.add_edge("collect_candidate_pool", "first_filter_candidates")
graph_builder.add_edge("first_filter_candidates", "second_filter_candidates")
graph_builder.add_edge("second_filter_candidates", "travel_matrix")
graph_builder.add_edge("travel_matrix", "plan_itinerary")
graph_builder.add_edge("plan_itinerary", END)

graph = graph_builder.compile()

In [ ]:
# 그래프 실행
state_v1 = await graph.ainvoke(initial_state)

print(f"📍 위치: {state_v1['user_input']['location']}")
print(f"📌 좌표: ({state_v1['user_input']['center_lat']}, {state_v1['user_input']['center_lng']})")
print(f"🔍 Kakao raw 후보: {len(state_v1['candidates'])}개")
print(f"⚠️  warnings: {state_v1['warnings']}")
print(f"❌ errors: {state_v1['errors']}")
print(f"✅ step: {state_v1['step']}\n")

for p in filter_result["filtered_candidates"][:5]:
    print(
        f"  [{p.get('category_group_code', '기타')}] {p['name']} - "
        f"{p.get('road_address_name') or p.get('address_name', '')}"
    )

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))